In [ ]:
"""
07_logistic_regression.py
"""

In [1]:
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd().resolve().parents[0] if Path.cwd(
).name == "notebooks" else Path.cwd()
sys.path.append(str(project_root))

df = pd.read_csv(str(project_root / "data" / "processed" / "model_features.csv"))

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

from utils.print_section import print_section

FEATURES = [
    "profitability",
    "liquidity",
    "solvency",
    "structure",
    "log_age",
    "size",
]
TARGET = "target"
X = df[FEATURES]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

pipeline = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "model",
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )
    )
])

pipeline.fit(
    X_train,
    y_train
)

y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:, 1]

print_section("Performance")

print(
    f"Accuracy : {accuracy_score(y_test, y_pred):.4f}"
)

print(
    f"Precision: {precision_score(y_test, y_pred):.4f}"
)

print(
    f"Recall   : {recall_score(y_test, y_pred):.4f}"
)

print(
    f"F1-score : {f1_score(y_test, y_pred):.4f}"
)

print(
    f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}"
)

print_section("Confusion matrix")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

print_section("Classification report")

print(
    classification_report(
        y_test,
        y_pred
    )
)

print_section("Coefficients")

coefs = pd.DataFrame({
    "feature": FEATURES,
    "coefficient": (
        pipeline.named_steps["model"]
        .coef_[0]
    )
})

coefs = coefs.sort_values(
    "coefficient"
)

print(coefs)


print_section("Baseline")

print(
    y.mean()
)


print(
    f"Failure rate: {y.mean():.4%}"
)


print(
    f"Failure rate: {y.mean():.4%}"
)

print_section("Model improvement over baseline")

print(
    f"Baseline failure rate: "
    f"{y.mean():.4%}"
)

print(
    f"Model precision: "
    f"{precision_score(y_test, y_pred):.4%}"
)

print(
    f"Improvement factor: "
    f"{precision_score(y_test, y_pred) / y.mean():.2f}x"
)


Performance
Accuracy : 0.8104
Precision: 0.0094
Recall   : 0.6259
F1-score : 0.0186
ROC-AUC  : 0.7939

Confusion matrix
[[158780  37020]
 [   211    353]]

Classification report
              precision    recall  f1-score   support

           0       1.00      0.81      0.90    195800
           1       0.01      0.63      0.02       564

    accuracy                           0.81    196364
   macro avg       0.50      0.72      0.46    196364
weighted avg       1.00      0.81      0.89    196364


Coefficients
         feature  coefficient
0  profitability    -1.280535
4        log_age    -0.179777
2       solvency    -0.159466
5           size    -0.065360
1      liquidity    -0.035583
3      structure     1.368426

Baseline
0.002874259791529591
Failure rate: 0.2874%
Failure rate: 0.2874%

Model improvement over baseline
Baseline failure rate: 0.2874%
Model precision: 0.9445%
Improvement factor: 3.29x
